# Kuiper kernel benchmarks

Each table compares the JIT-dispatched verified Kuiper kernels (`kuipy.run`) and
the unverified reference kernels (`kuipy.unverified`) against stock PyTorch, on
the shapes of a Qwen2.5-0.5B decode step.

Times are us/call, `rel-err` is the relative Frobenius norm against the `ref` column.

In [1]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

import kuipy
from kuipy import unverified
from kuipy.benchmarking import bench_matrix

aten = torch.ops.aten
DEV = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Qwen2.5-0.5B-Instruct, decoding at batch 256.
HID, NH, NKV, HEAD_DIM = 896, 14, 2, 64
INTER, VOCAB, BATCH = 4864, 151936, 256
SCALE = HEAD_DIM ** -0.5
ALPHA, BETA = 0.75, 1.5

_g = torch.Generator(device=DEV).manual_seed(0)

def rand(*shape, dtype=torch.bfloat16):
    return torch.randn(*shape, device=DEV, dtype=dtype, generator=_g) * 0.1

torch.cuda.get_device_name(0)

'NVIDIA RTX A6000'

## mm

`C = A @ B`. The unverified GEMMs are addmm-shaped, so they show up in the next section.

In [2]:
MM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("gate_proj",   BATCH, HID,   INTER),
    ("up_proj",     BATCH, HID,   INTER),
    ("down_proj",   BATCH, INTER, HID),
    ("lm_head",     BATCH, HID,   VOCAB),
    ("square_4096", 4096,  4096,  4096),
]

def mm_inputs(dtype):
    return lambda M, K, N: ((rand(M, K, dtype=dtype), rand(K, N, dtype=dtype)), {})

MNK = lambda M, K, N: (M, K, N)
GEMM_FLOPS = lambda M, K, N: 2 * M * N * K

bench_matrix(MM_CASES, mm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("tc2d_to", kuipy.run(aten.mm.default, impl="tc2d_to"))],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,tc2d_to GFLOP/s,ref GFLOP/s,tc2d_to us,ref us,tc2d_to rel-err
0,o_proj,256,896,896,455.007933,32737.933156,903.372803,12.555521,0.000000
1,gate_proj,256,896,4864,3922.298273,59407.633745,568.893433,37.560320,0.002713
2,up_proj,256,896,4864,3237.944542,59602.627469,689.131546,37.437439,0.002711
3,down_proj,256,4864,896,3016.027136,65595.185267,739.837418,34.017279,0.002622
4,lm_head,256,896,151936,15253.866587,102755.541794,4569.395142,678.318100,0.000000
5,square_4096,4096,4096,4096,18232.635868,119264.359683,7538.073730,1152.389145,0.000000


In [3]:
bench_matrix(MM_CASES, mm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("tc2d_to", kuipy.run(aten.mm.default, impl="tc2d_to")),
              ("tc2d", kuipy.run(aten.mm.default, impl="tc2d",
                                 acc_dtype=torch.float16))],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,tc2d_to GFLOP/s,tc2d GFLOP/s,ref GFLOP/s,tc2d_to us,tc2d us,ref us,tc2d_to rel-err,tc2d rel-err
0,o_proj,256,896,896,484.102348,416.415622,32848.445395,849.080353,987.095032,12.513280,0.000000,0.001573
1,gate_proj,256,896,4864,3944.878451,3943.745157,59342.919055,565.637131,565.799675,37.601280,0.000340,0.001588
2,up_proj,256,896,4864,4036.963219,4058.920358,59635.250941,552.734718,549.744644,37.416959,0.000339,0.001584
3,down_proj,256,4864,896,3487.407983,4034.720812,65437.598035,639.836159,553.041916,34.099200,0.000329,0.003622
4,lm_head,256,896,151936,11887.294266,61787.269000,101690.161943,5863.482666,1128.079376,685.424652,0.000000,0.001575
5,square_4096,4096,4096,4096,12783.834379,85690.948543,113239.062838,10750.996094,1603.891144,1213.706207,0.000000,0.003331


## addmm

`D = beta*C + alpha*(A @ B)`: the full epilogue, which the bias-free `mm` path never hits.
`gemm_pipe` is fp16-only and `gemm_hacky_epilogue` bf16-only, hence the two tables.

In [4]:
GEMM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("down_proj",   BATCH, INTER, HID),
    ("square_4096", 4096,  4096,  4096),
]

def addmm_inputs(dtype):
    return lambda M, K, N: ((rand(M, N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(K, N, dtype=dtype)), {"beta": BETA, "alpha": ALPHA})

bench_matrix(GEMM_CASES, addmm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("hacky_epilogue", unverified.gemm_hacky_epilogue)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,hacky_epilogue GFLOP/s,ref GFLOP/s,kuiper us,hacky_epilogue us,ref us,kuiper rel-err,hacky_epilogue rel-err
0,o_proj,256,896,896,339.474137,5975.111701,19372.973490,1210.819168,68.792319,21.217279,0.000004,0.000004
1,down_proj,256,4864,896,2230.877162,5958.011646,57739.055190,1000.220795,374.515839,38.645761,0.002559,0.002559
2,square_4096,4096,4096,4096,19684.299525,42855.597743,106842.530319,6982.161255,3207.024536,1286.369324,0.000005,0.000005


In [5]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("gemm_pipe", unverified.gemm_pipe)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,gemm_pipe GFLOP/s,ref GFLOP/s,kuiper us,gemm_pipe us,ref us,kuiper rel-err,gemm_pipe rel-err
0,o_proj,256,896,896,416.303326,9790.438976,19488.190067,987.361298,41.984000,21.091840,8.420163e-07,8.420163e-07
1,down_proj,256,4864,896,2213.829112,13309.747702,57834.835219,1007.923203,167.649288,38.581760,3.200818e-04,3.200818e-04
2,square_4096,4096,4096,4096,12267.835972,70302.714550,101055.389568,11203.194580,1954.959412,1360.035858,1.462452e-06,1.462452e-06


## sdpa

The Kuiper kernel's mask is a dense `(B, Hq, Sq, Sk)` tensor -- a Kuiper tlayout is an
injection, so there is no broadcast layout to instantiate it with -- so the mask is
materialised and handed to every contender for fairness.

In [6]:
_kuiper_sdpa = kuipy.run(aten._scaled_dot_product_efficient_attention.default)

def kuiper_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    return _kuiper_sdpa(q, k, v, attn_mask, False, 0.0, is_causal, scale=scale)[0]

def cudnn_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    with sdpa_kernel(SDPBackend.CUDNN_ATTENTION):
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                              is_causal=is_causal, scale=scale,
                                              enable_gqa=True)

def attn_flops(sq, sk):
    return 4 * BATCH * NH * sq * sk * HEAD_DIM

In [7]:
DECODE_CASES = [(f"ctx_{c}", 1, c) for c in (128, 512, 1024, 16384)]

def decode_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"attn_mask": torch.zeros(BATCH, NH, sq, sk, device=DEV,
                                      dtype=torch.bfloat16),
             "scale": SCALE})

bench_matrix(DECODE_CASES, decode_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("manual_extract", unverified.flash_attn_manual_extract),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,kuiper GFLOP/s,manual_extract GFLOP/s,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,kuiper us,manual_extract us,fa1 us,fa2 us,ref us,kuiper rel-err,manual_extract rel-err,fa1 rel-err,fa2 rel-err
0,ctx_128,1,128,616.072190,617.797864,681.449826,1886.936436,2340.571401,190.627842,190.095367,172.339191,62.238722,50.176001,0.002298,0.002298,0.002298,0.001673
1,ctx_512,1,512,1249.452257,1251.307520,1216.074665,3404.720281,3833.855930,375.974388,375.416946,386.293755,137.973757,122.529917,0.002333,0.002333,0.002333,0.001614
2,ctx_1024,1,1024,1400.897773,1432.346661,1337.430334,3695.142902,3965.012934,670.658569,655.933456,702.484512,254.259205,236.953602,0.002346,0.002346,0.002346,0.001487
3,ctx_16384,1,16384,1538.439272,1538.974723,1398.772671,3978.595930,4045.454453,9771.192017,9767.792358,10746.839600,3778.314209,3715.870667,0.002350,0.002350,0.002350,0.000879


In [8]:
# Prefill: full self-attention, is_causal, no explicit mask -- which the Kuiper
# kernel cannot express (see above), so only the unverified kernels compete.
PREFILL_CASES = [(f"seq_{s}", s, s) for s in (128, 512)]

def prefill_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"is_causal": True, "scale": SCALE})

bench_matrix(PREFILL_CASES, prefill_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,fa1 us,fa2 us,ref us,fa1 rel-err,fa2 rel-err
0,seq_128,128,128,5292.287097,16321.338164,65077.517813,2840.432739,921.026535,230.991993,0.000927,0.000927
1,seq_512,512,512,9641.373783,36482.973768,119998.884680,24946.462402,6592.614136,2004.336700,0.001074,0.001074
